# 03 — Train U-Net (ResNet-34, SMP) on Sen1Floods11

Trains `smp.Unet(encoder_name='resnet34', in_channels=2, classes=1)` on a free Colab T4 GPU in under ~30 minutes:
- Normalization computed strictly from training split
- Masked BCE + Dice loss ignoring `-1` no-data pixels
- Random 256x256 crops, flips, 90-degree rotations during training
- AdamW + CosineAnnealingLR + AMP mixed precision + early stopping on validation IoU
- Saves `ml/checkpoints/best_model.pt`, `training_log.json`, and `model_card.json` (plus copies to `backend/data/`)

In [ ]:
import json, sys
from pathlib import Path
import torch

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('CUDA available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

from ml import config
from ml.src.train import train_unet

baseline_path = config.CHECKPOINTS_DIR / 'baseline_metrics.json'
baseline_results = json.loads(baseline_path.read_text()) if baseline_path.exists() else None

model_card = train_unet(
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    lr=config.LEARNING_RATE,
    baseline_results=baseline_results,
)
print('\n=== Final Model Card Metrics ===')
for split_name in ('val', 'test', 'bolivia'):
    m = model_card['metrics'][split_name]
    print(f"{split_name:8s} | IoU={m['iou']:.4f} | F1={m['f1']:.4f} | CM={m['confusion_matrix']}")

In [ ]:
# Plot training loss and validation IoU curves
import matplotlib.pyplot as plt

log = json.loads((config.CHECKPOINTS_DIR / 'training_log.json').read_text())
epochs = [r['epoch'] for r in log]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [r['train_loss'] for r in log], label='Train Loss')
axes[0].plot(epochs, [r['val_loss'] for r in log], label='Val Loss')
axes[0].set_title('BCE + Dice Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, [r['val_iou'] for r in log], label='Val IoU')
axes[1].plot(epochs, [r['val_f1'] for r in log], label='Val F1')
axes[1].set_title('Validation Water Segmentation Metrics'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()